# Tensor Product Benchmarks

Here I compare tensor product variants on random inputs (closer to what you have in your code):
- e3nn FullyConnectedTensorProduct
- SeparateWeightTensorProduct (from the Mandala code)
- SO2OpsEquiformerDirect (using EquiformerV2 code)
- SO2OpsEquiformerMasked (same as above but with some masking to avoid loops)



In [18]:
import time
import torch
from e3nn.o3 import Irreps, FullyConnectedTensorProduct, spherical_harmonics

import importlib
import net.common as _common
import net.so2_ops_equiformer_direct_min as _so2_eq
import net.so2_ops_equiformer_direct_masked as _so2_eq_mask
import external.equiformer_v2.so2_ops as _eq_so2
import external.equiformer_v2.so3 as _eq_so3
import external.equiformer_v2.edge_rot_mat as _eq_rot
importlib.reload(_eq_so2)
importlib.reload(_eq_so3)
importlib.reload(_eq_rot)
importlib.reload(_common)
importlib.reload(_so2_eq)
importlib.reload(_so2_eq_mask)
from net.common import SeparateWeightTensorProduct, RotatedTensorProduct
from net.so2_ops_equiformer_direct_min import SO2OpsEquiformerDirect
from net.so2_ops_equiformer_direct_masked import SO2OpsEquiformerDirectMasked
from external.equiformer_v2.so2_ops import SO2_Convolution
from external.equiformer_v2.so3 import SO3_Embedding, CoefficientMappingModule, SO3_Rotation
from external.equiformer_v2.edge_rot_mat import init_edge_rot_mat

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision('high')
    except Exception:
        pass
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(0)
device


device(type='cuda')

In [19]:
def _alt_irreps_str(l_max, base_mul, start_parity):
    parts = []
    mul = base_mul
    parity = start_parity
    for l in range(l_max + 1):
        p = 'e' if parity == 1 else 'o'
        parts.append(f"{mul}x{l}{p}")
        mul = max(1, mul // 2)
        parity *= -1
    return ' + '.join(parts)

L_small = 4
irreps_in1 = Irreps(_alt_irreps_str(L_small, 32, -1))
irreps_in2 = Irreps.spherical_harmonics(L_small)
irreps_out = Irreps(_alt_irreps_str(L_small, 16, -1))

batch = 4096
x1 = irreps_in1.randn(batch, -1).to(device)
r_hat = torch.randn(batch, 3, device=device)
r_hat = r_hat / r_hat.norm(dim=-1, keepdim=True)
x2 = spherical_harmonics(irreps_in2, r_hat, normalize=True, normalization='component')
x2 = x2.to(device)

edge_dim = 32 #not sure about this one
edge_emb = torch.randn(batch, edge_dim, device=device)

x1.shape, x2.shape, r_hat.shape, edge_emb.shape


(torch.Size([4096, 166]),
 torch.Size([4096, 25]),
 torch.Size([4096, 3]),
 torch.Size([4096, 32]))

In [20]:
tp_fc = FullyConnectedTensorProduct(irreps_in1, irreps_in2, irreps_out).to(device)
tp_sw = SeparateWeightTensorProduct(irreps_in1, irreps_in2, irreps_out).to(device)
tp_so2_eq = SO2OpsEquiformerDirect(
    irreps_in1,
    irreps_out,
    edge_dim=edge_dim,
    max_m=2,
).to(device)
tp_so2_eq_mask = SO2OpsEquiformerDirectMasked(
    irreps_in1,
    irreps_out,
    edge_dim=edge_dim,
    max_m=2,
).to(device)

with torch.no_grad():
    y_fc = tp_fc(x1, x2)
    y_sw = tp_sw(x1, x2)
    y_so2_eq = tp_so2_eq(x1, r_hat, edge_emb)
    y_so2_eq_mask = tp_so2_eq_mask(x1, r_hat, edge_emb)

print('shapes:', y_fc.shape, y_sw.shape, y_so2_eq.shape, y_so2_eq_mask.shape)
print('so2 equiformer max abs:', y_so2_eq.abs().max().item())
print('so2 equiformer masked max abs:', y_so2_eq_mask.abs().max().item())


shapes: torch.Size([4096, 83]) torch.Size([4096, 83]) torch.Size([4096, 83]) torch.Size([4096, 83])
so2 equiformer max abs: 0.6927645206451416
so2 equiformer masked max abs: 0.6774137616157532


In [21]:
# Torch compile optimizations
use_compile = True  
compile_backend = 'inductor'  

import os
import shutil


if use_compile:
    try:
        import torch._dynamo
        torch._dynamo.config.suppress_errors = True
    except Exception:
        pass

def _select_backend():
    backend = compile_backend
    if backend == 'inductor' and os.name == 'nt':
        if shutil.which('cl') is None:
            print('MSVC compiler (cl) not found; using backend="eager" instead')
            backend = 'eager'
    return backend

def _maybe_compile(module, name):
    if not use_compile:
        return module
    if not hasattr(torch, 'compile'):
        print(f'torch.compile not available for {name}')
        return module
    backend = _select_backend()
    try:
        compiled = torch.compile(module, backend=backend)
        print(f'torch.compile enabled for {name} (backend={backend})')
        return compiled
    except Exception as e:
        print(f'torch.compile failed for {name}: {type(e).__name__}: {e}')
        return module

tp_fc = _maybe_compile(tp_fc, 'tp_fc')
tp_sw = _maybe_compile(tp_sw, 'tp_sw')
tp_so2_eq = _maybe_compile(tp_so2_eq, 'tp_so2_eq')
tp_so2_eq_mask = _maybe_compile(tp_so2_eq_mask, 'tp_so2_eq_mask')


MSVC compiler (cl) not found; using backend="eager" instead
torch.compile enabled for tp_fc (backend=eager)
MSVC compiler (cl) not found; using backend="eager" instead
torch.compile enabled for tp_sw (backend=eager)
MSVC compiler (cl) not found; using backend="eager" instead
torch.compile enabled for tp_so2_eq (backend=eager)
MSVC compiler (cl) not found; using backend="eager" instead
torch.compile enabled for tp_so2_eq_mask (backend=eager)


# Sanity Check for simple Rotation

In [22]:
tp_rot_ref = RotatedTensorProduct(irreps_in1, irreps_in2, irreps_out, tp=tp_fc).to(device)
with torch.no_grad():
    y_rot_ref = tp_rot_ref(x1, x2, r_hat)

max_abs_ref = (y_fc - y_rot_ref).abs().max().item()
print('rotated wrapper vs fc max abs:', max_abs_ref)
if max_abs_ref > 1e-4:
    print('warning: rotated wrapper differs from fc beyond 1e-4')

print('fc vs so2-eq max abs:', (y_fc - y_so2_eq).abs().max().item())
print('fc vs so2-eq-mask max abs:', (y_fc - y_so2_eq_mask).abs().max().item())
print('sw vs so2-eq max abs:', (y_sw - y_so2_eq).abs().max().item())
print('sw vs so2-eq-mask max abs:', (y_sw - y_so2_eq_mask).abs().max().item())


rotated wrapper vs fc max abs: 0.0026859045028686523
fc vs so2-eq max abs: 5.936307907104492
fc vs so2-eq-mask max abs: 6.22566556930542
sw vs so2-eq max abs: 8.804046630859375
sw vs so2-eq-mask max abs: 8.916468620300293


In [23]:
benchmark_mode = 'inference'  
do_backward = False           # true if "train"
use_cuda_events = True        

def _wrap_step(fn, module):
    if benchmark_mode == 'inference':
        module.eval()
        def run():
            with torch.no_grad():
                return fn()
        return run
    module.train()
    def run():
        out = fn()
        if do_backward:
            out.sum().backward()
            module.zero_grad(set_to_none=True)
        return out
    return run

def bench(fn, iters=50, warmup=10):
    if device.type == 'cuda' and use_cuda_events:
        starter = torch.cuda.Event(enable_timing=True)
        ender = torch.cuda.Event(enable_timing=True)
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        times = []
        for _ in range(iters):
            starter.record()
            fn()
            ender.record()
            torch.cuda.synchronize()
            times.append(starter.elapsed_time(ender))
        return (sum(times) / len(times)) / 1000.0
    else:
        for _ in range(warmup):
            fn()
        if device.type == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(iters):
            fn()
        if device.type == 'cuda':
            torch.cuda.synchronize()
        return (time.perf_counter() - t0) / iters

t_fc = bench(_wrap_step(lambda: tp_fc(x1, x2), tp_fc))
t_sw = bench(_wrap_step(lambda: tp_sw(x1, x2), tp_sw))
t_so2_eq = bench(_wrap_step(lambda: tp_so2_eq(x1, r_hat, edge_emb), tp_so2_eq))
t_so2_eq_mask = bench(_wrap_step(lambda: tp_so2_eq_mask(x1, r_hat, edge_emb), tp_so2_eq_mask))

print(f'FullyConnectedTensorProduct: {t_fc * 1e3:.3f} ms')
print(f'SeparateWeightTensorProduct:  {t_sw * 1e3:.3f} ms')
print(f'SO2OpsEquiformerDirect: {t_so2_eq * 1e3:.3f} ms')
print(f'SO2OpsEquiformerMasked: {t_so2_eq_mask * 1e3:.3f} ms')


FullyConnectedTensorProduct: 6.477 ms
SeparateWeightTensorProduct:  9.955 ms
SO2OpsEquiformerDirect: 23.808 ms
SO2OpsEquiformerMasked: 24.273 ms


# Tests for higher L

In [24]:
run_big = True
if run_big:
    l_max_big = 10
    irreps_in1_big = Irreps(_alt_irreps_str(l_max_big, 64, -1))
    irreps_in2_big = Irreps.spherical_harmonics(l_max_big)
    irreps_out_big = Irreps(_alt_irreps_str(l_max_big, 32, -1))

    batch_big = 4096
    x1_big = irreps_in1_big.randn(batch_big, -1).to(device)
    r_hat_big = torch.randn(batch_big, 3, device=device)
    r_hat_big = r_hat_big / r_hat_big.norm(dim=-1, keepdim=True)
    x2_big = spherical_harmonics(irreps_in2_big, r_hat_big, normalize=True, normalization='component')
    x2_big = x2_big.to(device)
    edge_emb_big = torch.randn(batch_big, edge_dim, device=device)

    tp_fc_big = FullyConnectedTensorProduct(irreps_in1_big, irreps_in2_big, irreps_out_big).to(device)
    tp_sw_big = SeparateWeightTensorProduct(irreps_in1_big, irreps_in2_big, irreps_out_big).to(device)
    tp_so2_eq_big = SO2OpsEquiformerDirect(
        irreps_in1_big,
        irreps_out_big,
        edge_dim=edge_dim,
        max_m=2,
    ).to(device)
    tp_so2_eq_mask_big = SO2OpsEquiformerDirectMasked(
        irreps_in1_big,
        irreps_out_big,
        edge_dim=edge_dim,
        max_m=2,
    ).to(device)



    def _bench_big(label, fn, module):
        print(f'bench {label} start', flush=True)
        try:
            t = bench(_wrap_step(fn, module))
        except Exception as e:
            print(f'bench {label} failed: {type(e).__name__}: {e}', flush=True)
            return None
        print(f'bench {label} done', flush=True)
        return t

    t_fc_big = _bench_big('fc', lambda: tp_fc_big(x1_big, x2_big), tp_fc_big)
    t_sw_big = _bench_big('sw', lambda: tp_sw_big(x1_big, x2_big), tp_sw_big)
    t_so2_eq_big = _bench_big('so2_eq', lambda: tp_so2_eq_big(x1_big, r_hat_big, edge_emb_big), tp_so2_eq_big)
    t_so2_eq_mask_big = _bench_big('so2_eq_mask', lambda: tp_so2_eq_mask_big(x1_big, r_hat_big, edge_emb_big), tp_so2_eq_mask_big)

    if t_fc_big is not None:
        print(f'Big L FullyConnectedTensorProduct: {t_fc_big * 1e3:.3f} ms')
    if t_sw_big is not None:
        print(f'Big L SeparateWeightTensorProduct:  {t_sw_big * 1e3:.3f} ms')
    if t_so2_eq_big is not None:
        print(f'Big L SO2OpsEquiformerDirect: {t_so2_eq_big * 1e3:.3f} ms')
    if t_so2_eq_mask_big is not None:
        print(f'Big L SO2OpsEquiformerMasked: {t_so2_eq_mask_big * 1e3:.3f} ms')


bench fc start
bench fc done
bench sw start
bench sw done
bench so2_eq start
bench so2_eq done
bench so2_eq_mask start
bench so2_eq_mask done
Big L FullyConnectedTensorProduct: 135.273 ms
Big L SeparateWeightTensorProduct:  187.822 ms
Big L SO2OpsEquiformerDirect: 104.092 ms
Big L SO2OpsEquiformerMasked: 97.238 ms


## Simple Equiformer SO(2) Benchmark 

Benchmark test outside of loop. Here we just pack from Irreps once then time rotate then apply SO2 then rotate_inv


In [25]:
eq_lmax = l_max_big
eq_mmax = 2
eq_channels = max(mul for mul, _ in irreps_in1_big)  
eq_mapping = CoefficientMappingModule([eq_lmax], [eq_mmax]).to(device)
eq_rotation = SO3_Rotation(eq_lmax).to(device)
eq_rotation.set_wigner(init_edge_rot_mat(r_hat_big))

def _pack_flat_irreps_to_eq(x_flat, irreps, lmax, channels):
    B = x_flat.shape[0]
    eq = torch.zeros(B, (lmax + 1) ** 2, channels, device=x_flat.device, dtype=x_flat.dtype)
    offset = 0
    for mul, ir in irreps:
        dim = ir.dim
        start = ir.l * ir.l
        blk = x_flat[:, offset : offset + mul * dim].view(B, mul, dim)
        for c in range(mul):
            eq[:, start : start + dim, c] = blk[:, c, :]
        offset += mul * dim
    return eq

eq_x_base = _pack_flat_irreps_to_eq(x1_big, irreps_in1_big, eq_lmax, eq_channels)
eq_so2 = SO2_Convolution(
    eq_channels,
    eq_channels,
    [eq_lmax],
    [eq_mmax],
    eq_mapping,
    internal_weights=False,
    edge_channels_list=[edge_dim, edge_dim],
).to(device)

def _eq_pure_step():
    x_rot = SO3_Embedding(0, [eq_lmax], eq_channels, device, eq_x_base.dtype)
    x_rot.set_embedding(eq_rotation.rotate(eq_x_base, eq_lmax, eq_mmax))
    x_rot.set_lmax_mmax([eq_lmax], [eq_mmax])
    y = eq_so2(x_rot, edge_emb_big)
    y._rotate_inv([eq_rotation], eq_mapping)
    return y.embedding

t_so2_pure = bench(_wrap_step(_eq_pure_step, eq_so2))
print(f'Big L SO2OpsPureEquiformer: {t_so2_pure * 1e3:.3f} ms')


Big L SO2OpsPureEquiformer: 15.397 ms


In [26]:
def nparams(module):
    return sum(p.numel() for p in module.parameters())

print('params (fc):', nparams(tp_fc))
print('params (sw):', nparams(tp_sw))
print('params (so2_eq):', nparams(tp_so2_eq))
print('params (so2_eq_mask):', nparams(tp_so2_eq_mask))


params (fc): 2382
params (sw): 2595
params (so2_eq): 63552
params (so2_eq_mask): 63552
